# Pré-processamento de Dados - Modelo de Inferência do Perfil de Aprendizagem (LP)

Este notebook tem como objetivo realizar as etapas de pré-processamento dos dados simulados para o modelo de inferência do perfil de aprendizagem. Isso inclui codificação de variáveis categóricas, escalamento de variáveis numéricas e divisão dos dados em conjuntos de treino e teste.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os

# Configurações para visualização (opcional, mas útil para verificar dados)
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

## 1. Carregamento dos Dados

In [2]:
data_path = os.path.join("..", "data", "raw", "simulated_lp_data.csv")
df = pd.read_csv(data_path)

print(f"Dataset carregado com {df.shape[0]} linhas e {df.shape[1]} colunas.")
df.head()

Dataset carregado com 2000 linhas e 14 colunas.


,time_spent_on_video,time_spent_on_audio,time_spent_on_reading,time_spent_on_writing,time_spent_on_quizz,time_spent_on_flashcards,time_spent_on_projects,completed_exercices,completed_quizzes,completed_flashcards,text_quizzes_accuracy,visual_quizzes_accuracy,most_preferred_resource_type,learning_profiles
0,7.358753,2.270760,1.409519,0.771837,1.950462,0.982258,4.349628,5.0,8,8,51.342521,64.024787,video,Auditivo
1,5.277661,3.262508,3.906390,1.738138,1.244525,1.541751,0.000000,0.0,0,10,85.010300,96.269294,practical,Auditivo
2,6.943066,1.415160,5.850360,1.241110,1.741438,1.231433,1.344410,10.0,13,19,98.501170,71.880268,practical,Leitura/Escrita
3,15.045823,2.384077,5.474606,1.904008,3.072098,2.165627,1.916680,12.0,4,19,96.774514,98.947800,practical,Visual
4,6.297400,0.000000,0.000000,2.571699,3.916785,3.795768,2.031817,11.0,13,10,69.714926,91.623119,video,Visual


## 2. Codificação do Target (`learning_profiles`)

In [3]:
le = LabelEncoder()
df["learning_profiles_encoded"] = le.fit_transform(df["learning_profiles"])

print("Mapeamento do LabelEncoder:")
for i, profile in enumerate(le.classes_):
    print(f"{profile}: {i}")

print("\nDistribuição do target codificado:")
print(df["learning_profiles_encoded"].value_counts())
df.head()

Mapeamento do LabelEncoder:
Auditivo: 0
Cinestésico: 1
Leitura/Escrita: 2
Visual: 3

Distribuição do target codificado:
learning_profiles_encoded
2    549
3    533
1    472
0    446
Name: count, dtype: int64


,time_spent_on_video,time_spent_on_audio,time_spent_on_reading,time_spent_on_writing,time_spent_on_quizz,time_spent_on_flashcards,time_spent_on_projects,completed_exercices,completed_quizzes,completed_flashcards,text_quizzes_accuracy,visual_quizzes_accuracy,most_preferred_resource_type,learning_profiles,learning_profiles_encoded
0,7.358753,2.270760,1.409519,0.771837,1.950462,0.982258,4.349628,5.0,8,8,51.342521,64.024787,video,Auditivo,0
1,5.277661,3.262508,3.906390,1.738138,1.244525,1.541751,0.000000,0.0,0,10,85.010300,96.269294,practical,Auditivo,0
2,6.943066,1.415160,5.850360,1.241110,1.741438,1.231433,1.344410,10.0,13,19,98.501170,71.880268,practical,Leitura/Escrita,2
3,15.045823,2.384077,5.474606,1.904008,3.072098,2.165627,1.916680,12.0,4,19,96.774514,98.947800,practical,Visual,3
4,6.297400,0.000000,0.000000,2.571699,3.916785,3.795768,2.031817,11.0,13,10,69.714926,91.623119,video,Visual,3


## 3. Codificação da Feature Categórica (`most_preferred_resource_type`)

In [4]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded_features = ohe.fit_transform(df[["most_preferred_resource_type"]])
encoded_df = pd.DataFrame(encoded_features, columns=ohe.get_feature_names_out(["most_preferred_resource_type"]))

df = pd.concat([df, encoded_df], axis=1)
df = df.drop("most_preferred_resource_type", axis=1)

print("Features categóricas codificadas:")
df.head()

Features categóricas codificadas:


,time_spent_on_video,time_spent_on_audio,time_spent_on_reading,time_spent_on_writing,time_spent_on_quizz,time_spent_on_flashcards,time_spent_on_projects,completed_exercices,completed_quizzes,completed_flashcards,text_quizzes_accuracy,visual_quizzes_accuracy,learning_profiles,learning_profiles_encoded,most_preferred_resource_type_audio,most_preferred_resource_type_practical,most_preferred_resource_type_text,most_preferred_resource_type_video
0,7.358753,2.270760,1.409519,0.771837,1.950462,0.982258,4.349628,5.0,8,8,51.342521,64.024787,Auditivo,0,0.0,0.0,0.0,1.0
1,5.277661,3.262508,3.906390,1.738138,1.244525,1.541751,0.000000,0.0,0,10,85.010300,96.269294,Auditivo,0,0.0,1.0,0.0,0.0
2,6.943066,1.415160,5.850360,1.241110,1.741438,1.231433,1.344410,10.0,13,19,98.501170,71.880268,Leitura/Escrita,2,0.0,1.0,0.0,0.0
3,15.045823,2.384077,5.474606,1.904008,3.072098,2.165627,1.916680,12.0,4,19,96.774514,98.947800,Visual,3,0.0,1.0,0.0,0.0
4,6.297400,0.000000,0.000000,2.571699,3.916785,3.795768,2.031817,11.0,13,10,69.714926,91.623119,Visual,3,0.0,0.0,0.0,1.0


## 4. Escalamento das Features Numéricas

In [5]:
numerical_features = df.select_dtypes(include=np.number).columns.tolist()
# Remover o target codificado da lista de features numéricas para escalamento
numerical_features.remove("learning_profiles_encoded")

scaler = StandardScaler()
df[numerical_features] = scaler.fit_transform(df[numerical_features])

print("Features numéricas escaladas:")
df.head()

Features numéricas escaladas:


,time_spent_on_video,time_spent_on_audio,time_spent_on_reading,time_spent_on_writing,time_spent_on_quizz,time_spent_on_flashcards,time_spent_on_projects,completed_exercices,completed_quizzes,completed_flashcards,text_quizzes_accuracy,visual_quizzes_accuracy,learning_profiles,learning_profiles_encoded,most_preferred_resource_type_audio,most_preferred_resource_type_practical,most_preferred_resource_type_text,most_preferred_resource_type_video
0,0.447868,-0.486736,-0.972866,-1.159249,-0.161270,-0.790188,0.649536,-0.820342,0.221626,-0.249431,-1.693160,-0.898350,Auditivo,0,-0.548079,-0.564262,-0.59583,1.663273
1,-0.167301,-0.036335,-0.201426,-0.721453,-0.641266,-0.398480,-1.117158,-1.541207,-1.633953,0.101756,0.628125,1.353518,Auditivo,0,-0.548079,1.772226,-0.59583,-0.601224
2,0.324991,-0.875306,0.399189,-0.946639,-0.303394,-0.615737,-0.571098,-0.099477,1.381362,1.682099,1.558277,-0.349744,Leitura/Escrita,2,-0.548079,1.772226,-0.59583,-0.601224
3,2.720159,-0.435273,0.283095,-0.646303,0.601376,0.038305,-0.338658,0.188869,-0.706163,1.682099,1.439230,1.540578,Visual,3,-0.548079,1.772226,-0.59583,-0.601224
4,0.134133,-1.518000,-1.408355,-0.343796,1.175714,1.179588,-0.291892,0.044696,1.381362,0.101756,-0.426442,1.029042,Visual,3,-0.548079,-0.564262,-0.59583,1.663273


## 5. Divisão dos Dados em Conjuntos de Treino e Teste

In [6]:
X = df.drop(["learning_profiles", "learning_profiles_encoded"], axis=1)
y = df["learning_profiles_encoded"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print(f"Shape de X_train: {X_train.shape}")
print(f"Shape de X_test: {X_test.shape}")
print(f"Shape de y_train: {y_train.shape}")
print(f"Shape de y_test: {y_test.shape}")

print("\nDistribuição do target em y_train:")
print(y_train.value_counts(normalize=True) * 100)
print("\nDistribuição do target em y_test:")
print(y_test.value_counts(normalize=True) * 100)

Shape de X_train: (1400, 16)
Shape de X_test: (600, 16)
Shape de y_train: (1400,)
Shape de y_test: (600,)

Distribuição do target em y_train:
learning_profiles_encoded
2    27.428571
3    26.642857
1    23.642857
0    22.285714
Name: proportion, dtype: float64

Distribuição do target em y_test:
learning_profiles_encoded
2    27.500000
3    26.666667
1    23.500000
0    22.333333
Name: proportion, dtype: float64


## 6. Salvar Dados Pré-processados

In [7]:
output_dir = os.path.join("..", "data", "processed")
os.makedirs(output_dir, exist_ok=True)

X_train.to_csv(os.path.join(output_dir, "X_train.csv"), index=False)
X_test.to_csv(os.path.join(output_dir, "X_test.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "y_train.csv"), index=False)
y_test.to_csv(os.path.join(output_dir, "y_test.csv"), index=False)

print(f"Dados de treino e teste salvos em {output_dir}")

Dados de treino e teste salvos em ../data/processed
